### Note: Make Sure NLTK datasets (punkt, stopwords) are installed before running this notebook.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

## Pre-processing

### 1. Converting to lowercase

In [8]:
df["review"] = df["review"].str.lower()

### 2. Removing The URLs

In [9]:
import re

# sample_text = "abc is the word, abc" # abc => xyz

# new_text = re.sub("abc", "xyz", sample_text)

In [10]:
# new_text 

'xyz is the word, xyz'

In [11]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)  # (pattern, repl, string) eg - https://www.google.com
    return text
    
df["review"] = df["review"].apply(remove_urls)

### 3. Removing Punctuations

In [12]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [13]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4. Removing HTML

In [14]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing The Stopwords

In [15]:
import nltk

# nltk.download("punkt")
# nltk.download("punkt_tab")
# nltk.download("stopwords")

In [16]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [17]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)

In [18]:
# tokens

['I', 'like', 'coding', 'in', 'python', '!']

In [19]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    
    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    
    return text

df["review"] = df["review"].apply(remove_stopwords)

In [20]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [21]:
# running -> run
# played -> play
# PorterStemming

from nltk.stem import PorterStemmer

In [22]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [23]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [24]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [25]:
y = df["sentiment"]

In [26]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [27]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## Dataset & Data Loaders

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [30]:
X_train.shape

(39665, 5000)

In [31]:
X_test.shape

(9917, 5000)

In [32]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3248304 stored elements and shape (39665, 5000)>

In [33]:
X_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 808836 stored elements and shape (9917, 5000)>

In [34]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [35]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [36]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [37]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build Our RNN

In [38]:
import torch.nn as nn
import torch.optim as optim

In [39]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        
        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep
        
        out = self.fc(out[:, -1, :])
        return out

In [40]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training The RNN

In [41]:
epochs = 10

for epoch in range(epochs):
    model.train()
    
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        
        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)
        
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability
        
        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update
    
    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.35938557982444763
epoch = 2/10 and loss = 0.2755436301231384
epoch = 3/10 and loss = 0.3512657582759857
epoch = 4/10 and loss = 0.2591031789779663
epoch = 5/10 and loss = 0.226608544588089
epoch = 6/10 and loss = 0.317380428314209
epoch = 7/10 and loss = 0.19997048377990723
epoch = 8/10 and loss = 0.20017272233963013
epoch = 9/10 and loss = 0.4100169241428375
epoch = 10/10 and loss = 0.17154039442539215


# Evaluate

In [42]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)
        
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
        
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()
    
    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.7719068266613
